# Exploratory work (early)

These notebooks are **early exploratory sketches** written before the pre-registered
evaluation. They do not produce any committed result and must not be read as the final
evaluation. The authoritative numbers are in `docs/RESULTS.md` and `results/*.json`.

Cell outputs have been cleared. Paths are relative to this folder, and
`data/processed/advanced_gold_features.csv` is **not redistributed** (decision D14),
so the cells will not run on a fresh clone without the full local dataset.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [ ]:
# 1. Load Data
df = pd.read_csv('../../data/processed/advanced_gold_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df = df.dropna() # Remove any NaN values from lag/rolling creations

In [ ]:
# 2. Select Features and Target
target_col = 'Target_Next_LogRet'
# Drop Date and the other target to prevent data leakage
features = df.drop(columns=['Date', 'Target_Next_Price', target_col]).columns.tolist()

In [ ]:
# 3. Scaling
# We scale features and target separately to easily inverse_transform later
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

scaled_features = feature_scaler.fit_transform(df[features])
scaled_target = target_scaler.fit_transform(df[[target_col]])

In [ ]:
# 4. Sequence Windowing Function
def create_sequences(features_data, target_data, lookback=14):
    X, y = [], []
    for i in range(len(features_data) - lookback):
        X.append(features_data[i:(i + lookback)])
        y.append(target_data[i + lookback])
    return np.array(X), np.array(y)

lookback_window = 14
X, y = create_sequences(scaled_features, scaled_target, lookback=lookback_window)

In [ ]:
# 5. Train / Val / Test Split (Time-based, NO Shuffle)
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")

In [ ]:
# 6. Build Baseline LSTM Model
model_baseline = Sequential([
    LSTM(50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(1)
])

model_baseline.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 7. Train Model with Early Stopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model_baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# 8. Plot Training History
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Baseline LSTM Loss')
plt.legend()
plt.show()

In [ ]:
from keras.layers import Bidirectional

# Build Advanced Bidirectional LSTM Model
model_bidi = Sequential([
    # Layer 1: Bidirectional LSTM with return_sequences=True to feed into the next LSTM
    Bidirectional(LSTM(64, activation='tanh', return_sequences=True), 
                  input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    
    # Layer 2: Second Bidirectional LSTM
    Bidirectional(LSTM(32, activation='tanh')),
    Dropout(0.3),
    
    # Dense layers for final output
    Dense(16, activation='relu'),
    Dense(1)
])

# Compile
model_bidi.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train Model with Early Stopping
early_stop_bidi = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)

print("Training Advanced Bidirectional LSTM...")
history_bidi = model_bidi.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_bidi],
    verbose=1
)

# Plot Training History
plt.figure(figsize=(8, 5))
plt.plot(history_bidi.history['loss'], label='Train Loss')
plt.plot(history_bidi.history['val_loss'], label='Val Loss')
plt.title('Advanced Bidirectional LSTM Loss')
plt.legend()
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import random as python_random
from keras.layers import GRU

np.random.seed(42)
python_random.seed(42)
tf.random.set_seed(42)

model_gru = Sequential([
    GRU(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    
    GRU(50),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop_gru = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

print("Training GRU Model...")
history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_gru],
    verbose=1
)

plt.figure(figsize=(8, 5))
plt.plot(history_gru.history['loss'], label='Train Loss')
plt.plot(history_gru.history['val_loss'], label='Val Loss')
plt.title('GRU Model Loss')
plt.legend()
plt.show()


In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

X_train_2d = X_train[:, -1, :] 
X_val_2d = X_val[:, -1, :]

import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

xgb_model = xgb.XGBRegressor(
    n_estimators=500,        
    learning_rate=0.05,      
    max_depth=5,             
    subsample=0.8,           
    colsample_bytree=0.8,    
    random_state=42,
    early_stopping_rounds=50  
)

print("Training XGBoost Model...")

xgb_model.fit(
    X_train_2d, y_train,
    eval_set=[(X_train_2d, y_train), (X_val_2d, y_val)],
    verbose=50                
)

y_pred_xgb = xgb_model.predict(X_val_2d)

mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))

print(f"\n--- XGBoost Results ---")
print(f"Validation MAE:  {mae_xgb:.5f}")
print(f"Validation RMSE: {rmse_xgb:.5f}")

plt.figure(figsize=(12, 5))
plt.plot(y_val[:100], label='Actual Target', marker='.')
plt.plot(y_pred_xgb[:100], label='XGBoost Prediction', marker='.')
plt.title('XGBoost: Actual vs Predicted (First 100 Val Days)')
plt.legend()
plt.show()



In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

print("--- Training LightGBM ---")

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=3,           
    num_leaves=7,          
    subsample=0.8,         
    colsample_bytree=0.8,  
    reg_alpha=0.1,         
    reg_lambda=0.5,        
    random_state=42
)

lgb_model.fit(
    X_train_2d, y_train,
    eval_set=[(X_train_2d, y_train), (X_val_2d, y_val)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

X_test_2d = X_test[:, -1, :]

lgb_preds = lgb_model.predict(X_test_2d)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_preds))
print(f"LightGBM Test RMSE: {lgb_rmse:.5f}")


In [ ]:
from catboost import CatBoostRegressor

print("\n--- Training CatBoost ---")

cb_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.01,
    depth=3,               
    l2_leaf_reg=5,         
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)


cb_model.fit(
    X_train_2d, y_train,
    eval_set=(X_val_2d, y_val),
    use_best_model=True
)

cb_preds = cb_model.predict(X_test_2d)
cb_rmse = np.sqrt(mean_squared_error(y_test, cb_preds))
print(f"CatBoost Test RMSE: {cb_rmse:.5f}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(y_test, label='Actual Target', color='blue', alpha=0.6)
plt.plot(lgb_preds, label='LightGBM Predictions', color='green', alpha=0.8)
plt.plot(cb_preds, label='CatBoost Predictions', color='red', alpha=0.8)

plt.title('Test Set Predictions: LightGBM vs CatBoost')
plt.xlabel('Time Steps (Test Data)')
plt.ylabel('Scaled Target_Next_LogRet')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np

print("--- Flattening 3D Data for Tree Models ---")

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"Old shape: {X_train.shape} -> New flattened shape: {X_train_flat.shape}")

y_train_1d = y_train.ravel()
y_val_1d = y_val.ravel()
y_test_1d = y_test.ravel()

import lightgbm as lgb
from sklearn.metrics import mean_squared_error

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=4,           
    num_leaves=15,         
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,        
    reg_lambda=0.1,        
    random_state=42
)

lgb_model.fit(
    X_train_flat, y_train_1d,
    eval_set=[(X_train_flat, y_train_1d), (X_val_flat, y_val_1d)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

lgb_preds = lgb_model.predict(X_test_flat)
lgb_rmse = np.sqrt(mean_squared_error(y_test_1d, lgb_preds))
print(f"\nLightGBM Test RMSE (Flattened Data): {lgb_rmse:.5f}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(y_test, label='Actual Target', color='blue', alpha=0.6)
plt.plot(lgb_preds, label='LightGBM Predictions', color='green', alpha=0.8)
plt.plot(cb_preds, label='CatBoost Predictions', color='red', alpha=0.8)

plt.title('Test Set Predictions: LightGBM vs CatBoost')
plt.xlabel('Time Steps (Test Data)')
plt.ylabel('Scaled Target_Next_LogRet')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import numpy as np

# ---------------------------------------------------------
# DIAGNOSTIC TEST: Forcing LightGBM to Overfit
# We are removing all regularizations to see if the model 
# is capable of finding any patterns in the flattened data.
# ---------------------------------------------------------

lgb_force_model = lgb.LGBMRegressor(
    n_estimators=500,        # Number of trees
    learning_rate=0.05,      # Higher learning rate for faster adaptation
    max_depth=8,             # Deeper trees to capture complex patterns
    num_leaves=127,          # Allow more leaf nodes (2^max_depth - 1 is max)
    min_child_samples=5,     # Allow splits even if a node has very few samples
    subsample=1.0,           # Use 100% of samples per tree
    colsample_bytree=1.0,    # Use 100% of features per tree
    reg_alpha=0.0,           # No L1 regularization
    reg_lambda=0.0,          # No L2 regularization
    random_state=42
)

# Train WITHOUT early stopping to force the model to build all 500 trees
print("Training Unrestricted LightGBM Model...")
lgb_force_model.fit(
    X_train_flat, y_train_1d,
    eval_set=[(X_train_flat, y_train_1d), (X_val_flat, y_val_1d)],
    eval_metric='rmse',
    callbacks=[lgb.log_evaluation(period=50)] # No early stopping callback
)

# Predict on Train and Test sets
train_preds = lgb_force_model.predict(X_train_flat)
test_preds = lgb_force_model.predict(X_test_flat)

# Print RMSE
train_rmse = np.sqrt(mean_squared_error(y_train_1d, train_preds))
test_rmse = np.sqrt(mean_squared_error(y_test_1d, test_preds))
print(f"\nTrain RMSE (Forced): {train_rmse:.5f}")
print(f"Test RMSE (Forced): {test_rmse:.5f}")

# ---------------------------------------------------------
# Plotting the results to visually inspect the predictions
# ---------------------------------------------------------
plt.figure(figsize=(14, 6))

# Plot only a slice of test data for better visibility (e.g., first 150 steps)
slice_len = min(150, len(y_test_1d))

plt.plot(y_test_1d[:slice_len], label='Actual Target', color='blue', alpha=0.6)
plt.plot(test_preds[:slice_len], label='Forced LightGBM Predictions', color='green', alpha=0.9)

plt.title('Diagnostic Test: Forced Unrestricted LightGBM')
plt.xlabel('Time Steps (Test Data Slice)')
plt.ylabel('Scaled Target_Next_LogRet')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
